# Pipeline de Preprocesamiento - Dataset UNSW-NB15

## Fase 1: Carga e Integracion de Datos (Data Integration)

En esta fase realizaremos:
1. Carga de los archivos `UNSW_NB15_training-set.csv` y `UNSW_NB15_testing-set.csv`.
2. Concatenacion de ambos conjuntos en un solo DataFrame.
3. Correccion del reinicio en la columna de identificadores (`id`).
4. Demostracion de los 4 tipos de `join` (`inner`, `left`, `right`, `outer`) usando `pd.merge` con una tabla auxiliar de severidad por categoria de ataque.

In [ ]:
# Paso 1.1: Importar librerias y cargar ambos archivos CSV
import pandas as pd
import numpy as np

ruta_train = "../data/UNSW_NB15_training-set.csv"
ruta_test = "../data/UNSW_NB15_testing-set.csv"

df_train = pd.read_csv(ruta_train)
df_test = pd.read_csv(ruta_test)

print("Dimensiones training-set:", df_train.shape)
print("Dimensiones testing-set:", df_test.shape)
print("Mismas columnas en ambos archivos:", list(df_train.columns) == list(df_test.columns))

df_train.head(3)

In [ ]:
# Paso 1.2: Concatenar ambos DataFrames en uno solo
df = pd.concat([df_train, df_test], ignore_index=True)

print("Dimensiones del dataset integrado:", df.shape)
print("Suma esperada de filas:", len(df_train) + len(df_test))
print("Coincide el total de filas:", len(df) == len(df_train) + len(df_test))

In [ ]:
# Paso 1.3: Detectar y corregir el reinicio de la columna id
# Ambos archivos originales empiezan su columna id desde 1, generando IDs repetidos
print("--- Antes de corregir la columna id ---")
print("Total de filas:", len(df))
print("IDs unicos:", df["id"].nunique())
print("IDs duplicados:", df["id"].duplicated().sum())

# Regenerar un identificador unico consecutivo del 1 al total de filas
df["id"] = range(1, len(df) + 1)

print("\n--- Despues de corregir la columna id ---")
print("IDs unicos:", df["id"].nunique())
print("ID minimo:", df["id"].min(), "| ID maximo:", df["id"].max())
print("IDs duplicados:", df["id"].duplicated().sum())

In [ ]:
# Paso 1.4: Crear una tabla auxiliar de referencia para demostrar los tipos de join
# Omitimos a proposito 'Worms' y agregamos 'ZeroDay' (que no existe en el dataset)
# para que cada tipo de join muestre diferencias claras en filas y valores nulos
print("Categorias presentes en df:", sorted(df["attack_cat"].unique()))

tabla_severidad_demo = pd.DataFrame({
    "attack_cat": [
        "Normal", "Reconnaissance", "Analysis", "Fuzzers",
        "DoS", "Generic", "Exploits", "Shellcode", "Backdoor", "ZeroDay"
    ],
    "nivel_severidad": [
        "Ninguna", "Baja", "Baja", "Media",
        "Alta", "Alta", "Alta", "Critica", "Critica", "Critica"
    ]
})

tabla_severidad_demo

In [ ]:
# Paso 1.5: Demostracion de inner, left, right y outer join con pd.merge
# Usamos solo 3 columnas de df para inspeccionar el resultado facilmente
df_base = df[["id", "attack_cat", "label"]]

join_inner = pd.merge(df_base, tabla_severidad_demo, on="attack_cat", how="inner")
join_left = pd.merge(df_base, tabla_severidad_demo, on="attack_cat", how="left")
join_right = pd.merge(df_base, tabla_severidad_demo, on="attack_cat", how="right")
join_outer = pd.merge(df_base, tabla_severidad_demo, on="attack_cat", how="outer")

print("INNER join -> filas:", len(join_inner), "| nulos en severidad:", join_inner["nivel_severidad"].isna().sum())
print("LEFT  join -> filas:", len(join_left), "| nulos en severidad:", join_left["nivel_severidad"].isna().sum())
print("RIGHT join -> filas:", len(join_right), "| nulos en id:", join_right["id"].isna().sum())
print("OUTER join -> filas:", len(join_outer), "| nulos en severidad:", join_outer["nivel_severidad"].isna().sum(), "| nulos en id:", join_outer["id"].isna().sum())

print("\nEjemplo de filas sin coincidencia en LEFT join (categoria Worms):")
print(join_left[join_left["nivel_severidad"].isna()].head(3))

print("\nEjemplo de fila sin coincidencia en RIGHT join (categoria ZeroDay):")
print(join_right[join_right["id"].isna()])

In [ ]:
# Paso 1.6: Tabla auxiliar completa (con las 10 categorias reales) y left join limpio
# Guardamos el resultado en df_con_severidad para analisis exploratorio,
# mientras df conserva las columnas originales para las siguientes fases de modelado
tabla_severidad_completa = pd.DataFrame({
    "attack_cat": [
        "Normal", "Reconnaissance", "Analysis", "Fuzzers",
        "DoS", "Generic", "Exploits", "Shellcode", "Backdoor", "Worms"
    ],
    "nivel_severidad": [
        "Ninguna", "Baja", "Baja", "Media",
        "Alta", "Alta", "Alta", "Critica", "Critica", "Critica"
    ]
})

df_con_severidad = pd.merge(df, tabla_severidad_completa, on="attack_cat", how="left")

print("Dimensiones de df_con_severidad:", df_con_severidad.shape)
print("Nulos en nivel_severidad:", df_con_severidad["nivel_severidad"].isna().sum())
print("\nDistribucion por nivel de severidad:")
print(df_con_severidad["nivel_severidad"].value_counts())

## Fase 2: Limpieza de Datos (Data Cleaning)

En esta fase realizaremos:
1. Deteccion y manejo de valores nulos (incluyendo valores ocultos como `'-'` en `service`).
2. Deteccion y eliminacion de registros duplicados (excluyendo `id`).
3. Estandarizacion de formatos de texto en columnas categoricas.
4. Deteccion de outliers con el metodo IQR y tratamiento mediante tope en percentil 99 (winsorizacion) con graficas de antes y despues.

In [ ]:
# Paso 2.1: Deteccion y manejo de valores nulos y valores ocultos ('-')
print("Nulos estandar (NaN) en todo el dataset:", df.isna().sum().sum())

# Revisar si existen guiones '-' usados como valores faltantes en columnas de texto
cols_texto = ["proto", "service", "state", "attack_cat"]
for col in cols_texto:
    cantidad_guion = (df[col] == "-").sum()
    print("Columna", col, "-> valores '-':", cantidad_guion)

# Convertir '-' a NaN en 'service' para tratarlo como nulo real
df["service"] = df["service"].replace("-", np.nan)
print("\nNulos detectados en 'service' tras reemplazar '-':", df["service"].isna().sum())

# Como mas del 50% de las filas tienen este valor faltante, NO eliminamos las filas;
# en su lugar imputamos con la categoria 'unknown'
df["service"] = df["service"].fillna("unknown")
print("Nulos en 'service' despues de imputar:", df["service"].isna().sum())
print("\nTop 5 categorias en 'service':")
print(df["service"].value_counts().head(5))

In [ ]:
# Paso 2.2: Deteccion y eliminacion de registros duplicados
# Si incluimos 'id', pandas no detecta duplicados porque cada 'id' es unico
print("Duplicados considerando 'id':", df.duplicated().sum())

# Evaluamos duplicados usando todas las columnas excepto 'id'
cols_sin_id = [col for col in df.columns if col != "id"]
duplicados_reales = df.duplicated(subset=cols_sin_id).sum()
print("Duplicados reales (excluyendo 'id'):", duplicados_reales)

filas_antes = len(df)
df = df.drop_duplicates(subset=cols_sin_id, keep="first").reset_index(drop=True)

# Regenerar 'id' consecutivo tras eliminar duplicados
df["id"] = range(1, len(df) + 1)

print("Filas antes:", filas_antes, "| Filas despues:", len(df))
print("Duplicados restantes (excluyendo 'id'):", df.duplicated(subset=cols_sin_id).sum())

In [ ]:
# Paso 2.3: Estandarizacion de formatos de texto en columnas categoricas
# Quitamos espacios sobrantes y pasamos proto, service y state a minusculas
for col in ["proto", "service", "state"]:
    df[col] = df[col].astype(str).str.strip().str.lower()

# En attack_cat quitamos espacios sobrantes manteniendo el formato legible
df["attack_cat"] = df["attack_cat"].astype(str).str.strip()

print("Valores unicos en state (estandarizado):", sorted(df["state"].unique()))
print("Valores unicos en service (estandarizado):", sorted(df["service"].unique()))
print("Valores unicos en attack_cat:", sorted(df["attack_cat"].unique()))
print("Total de protocolos unicos en proto:", df["proto"].nunique())

In [ ]:
# Paso 2.4: Deteccion de outliers usando el metodo IQR
import matplotlib.pyplot as plt
import seaborn as sns

# Seleccionamos variables numericas clave de trafico de red con valores extremos
cols_outliers = ["dur", "sbytes", "dbytes", "rate", "sload", "dload"]

print("Resumen de outliers detectados por regla IQR (Q3 + 1.5 * IQR):")
for col in cols_outliers:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    limite_inf = q1 - 1.5 * iqr
    limite_sup = q3 + 1.5 * iqr
    cantidad = ((df[col] < limite_inf) | (df[col] > limite_sup)).sum()
    porcentaje = (cantidad / len(df)) * 100
    print(f"{col:8s} -> outliers: {cantidad:6d} ({porcentaje:5.2f}%) | max actual: {df[col].max():.2f}")

In [ ]:
# Paso 2.5: Tratamiento de outliers (tope en percentil 99) y graficas antes vs despues
# En deteccion de intrusiones no eliminamos filas con outliers porque pueden ser ataques reales.
# En su lugar aplicamos un tope (clipping) en el percentil 99 para reducir valores extremos.
df_antes = df[["dur", "sbytes"]].copy()

for col in cols_outliers:
    limite_p99 = df[col].quantile(0.99)
    df[col] = df[col].clip(upper=limite_p99)

print("Maximos despues de aplicar tope en percentil 99:")
for col in cols_outliers:
    print(f"{col:8s} -> max nuevo: {df[col].max():.2f}")

# Graficas comparativas simples (antes y despues) para dur y sbytes
fig, axes = plt.subplots(2, 2, figsize=(12, 7))

sns.boxplot(x=df_antes["dur"], ax=axes[0, 0], color="salmon")
axes[0, 0].set_title("dur - Antes del tope (p99)")

sns.boxplot(x=df["dur"], ax=axes[0, 1], color="skyblue")
axes[0, 1].set_title("dur - Despues del tope (p99)")

sns.boxplot(x=df_antes["sbytes"], ax=axes[1, 0], color="salmon")
axes[1, 0].set_title("sbytes - Antes del tope (p99)")

sns.boxplot(x=df["sbytes"], ax=axes[1, 1], color="skyblue")
axes[1, 1].set_title("sbytes - Despues del tope (p99)")

plt.tight_layout()
plt.show()

## Fase 3: Transformacion de Datos (Data Transformation)

En esta fase realizaremos:
1. Optimizacion basica de tipos de datos para reducir uso de memoria.
2. Feature Engineering simple de trafico de red (`total_pkts`, `total_bytes`, `bytes_ratio`).
3. Codificacion de variables categoricas (`proto`, `service`, `state`) con One-Hot Encoding.
4. Preparacion de escalado (`StandardScaler` y `MinMaxScaler`) aplicando correctamente `train_test_split` antes del escalado para evitar *data leakage*.

In [ ]:
# Paso 3.1: Cambio y optimizacion basica de tipos de datos
mem_antes = df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"Memoria antes de optimizar: {mem_antes:.2f} MB")

# Reducir columnas enteras al tipo mas pequeno que soporte sus valores sin desbordarse
cols_enteras = df.select_dtypes(include=["int64"]).columns
for col in cols_enteras:
    df[col] = pd.to_numeric(df[col], downcast="integer")

# Reducir columnas decimales de float64 a float32
cols_flotantes = df.select_dtypes(include=["float64"]).columns
for col in cols_flotantes:
    df[col] = df[col].astype("float32")

# Convertir columnas de texto repetitivo a tipo category
cols_categoricas = ["proto", "service", "state", "attack_cat"]
for col in cols_categoricas:
    df[col] = df[col].astype("category")

mem_despues = df.memory_usage(deep=True).sum() / (1024 ** 2)
ahorro = ((mem_antes - mem_despues) / mem_antes) * 100
print(f"Memoria despues de optimizar: {mem_despues:.2f} MB (ahorro: {ahorro:.1f}%)")
print("\nResumen de tipos de datos actuales:")
print(df.dtypes.value_counts())

In [ ]:
# Paso 3.2: Creacion de nuevas variables (Feature Engineering simple de trafico de red)
# 1) Total de paquetes intercambiados (origen + destino)
df["total_pkts"] = (df["spkts"].astype("int32") + df["dpkts"].astype("int32")).astype("int32")

# 2) Total de bytes transferidos (origen + destino)
df["total_bytes"] = (df["sbytes"].astype("float32") + df["dbytes"].astype("float32")).astype("float32")

# 3) Proporcion de bytes enviados por el origen (sumamos 1 al denominador para evitar dividir por 0)
df["bytes_ratio"] = (df["sbytes"].astype("float32") / (df["total_bytes"] + 1.0)).astype("float32")

print("Nuevas columnas creadas: total_pkts, total_bytes, bytes_ratio")
print("Nulos en nuevas columnas:", df[["total_pkts", "total_bytes", "bytes_ratio"]].isna().sum().to_dict())
df[["spkts", "dpkts", "total_pkts", "sbytes", "dbytes", "total_bytes", "bytes_ratio"]].describe().round(3)

In [ ]:
# Paso 3.3: Separacion de variables (X, y) y codificacion de variables categoricas
# Excluimos 'id' (identificador), 'label' (objetivo binario) y 'attack_cat' (objetivo multiclase)
X = df.drop(columns=["id", "label", "attack_cat"]).copy()
y = df["label"].copy()

# 'proto' tiene mas de 130 protocolos, la mayoria muy poco frecuentes.
# Conservamos los 10 protocolos mas comunes y agrupamos el resto en 'other'
top_protos = X["proto"].value_counts().head(10).index.tolist()
X["proto"] = X["proto"].astype(str)
X["proto"] = X["proto"].where(X["proto"].isin(top_protos), "other")

print("Top 10 protocolos conservados:", top_protos)
print("Columnas de X antes de One-Hot Encoding:", X.shape[1])

# Aplicar One-Hot Encoding a proto, service y state
X_encoded = pd.get_dummies(X, columns=["proto", "service", "state"], drop_first=True, dtype=int)

print("Columnas de X despues de One-Hot Encoding:", X_encoded.shape[1])
print("Ejemplo de columnas nuevas:", [c for c in X_encoded.columns if c.startswith(("proto_", "service_", "state_"))][:10])

In [ ]:
# Paso 3.4: Escalado (StandardScaler y MinMaxScaler) SIN data leakage
# REGLA CLAVE: El train_test_split SIEMPRE va ANTES de ajustar (fit) el escalador.
# Si ajustamos el scaler con todo el dataset, informacion de prueba se filtra al entrenamiento.
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# 1) Dividir en entrenamiento (80%) y prueba (20%) manteniendo proporcion de clases (stratify=y)
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

print("Dimensiones X_train:", X_train.shape, "| X_test:", X_test.shape)

# 2) Opcion A: StandardScaler (media = 0, desviacion estandar = 1)
# fit_transform SOLO en X_train; transform (sin fit) en X_test
scaler_std = StandardScaler()
X_train_std = pd.DataFrame(
    scaler_std.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)
X_test_std = pd.DataFrame(
    scaler_std.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

# 3) Opcion B: MinMaxScaler (rango entre 0 y 1)
scaler_mm = MinMaxScaler()
X_train_mm = pd.DataFrame(
    scaler_mm.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)
X_test_mm = pd.DataFrame(
    scaler_mm.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

cols_check = ["dur", "sbytes", "total_bytes", "bytes_ratio"]
print("\nVerificacion StandardScaler en X_train (media ~0, std ~1):")
print(X_train_std[cols_check].agg(["mean", "std"]).round(3))

print("\nVerificacion MinMaxScaler en X_train (min = 0, max = 1):")
print(X_train_mm[cols_check].agg(["min", "max"]).round(3))

# Grafica rapida para comparar distribucion de sbytes antes y despues de StandardScaler
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(X_train["sbytes"], bins=40, ax=axes[0], color="salmon")
axes[0].set_title("sbytes (X_train) - Escala original")

sns.histplot(X_train_std["sbytes"], bins=40, ax=axes[1], color="skyblue")
axes[1].set_title("sbytes (X_train) - StandardScaler")

plt.tight_layout()
plt.show()